# Dating Agent RL — Tabular Q-Learning, SARSA, and Cached LLM World Model

This is the current active working notebook. Older versions are archived in `../archive/`.


This notebook is a toy **tabular reinforcement learning** project. The goal is to learn a policy over simplified conversation states, not to build real dating advice.

The discrete state is:

```text
interest x stage x tone
```

The action is an abstract conversation strategy. The RL agent does **not** generate natural language. It chooses one of these abstract actions:

```text
ask_question, give_compliment, share_story, be_playful,
be_direct, suggest_date, slow_down, end_chat
```

The LLM is optional and can be used in two separate ways:

A. **LLM classifier**

```text
real message -> interest/stage/tone
```

B. **LLM world model**

```text
state + action -> simulated next_state + response + outcome
```

The LLM is **not trained** in this notebook. The trained model is the Q-table or SARSA table.

Training loop:

```text
state + action
-> environment/world model
-> next_state + reward + done
-> TD update
-> Q-table changes
```

Interface loop:

```text
real message
-> LLM classifier
-> discrete RL state
-> trained Q-table
-> recommended abstract action
```

Cached LLM workflow:

```text
LLM generates transitions once
-> transitions saved to jsonl
-> cached environment samples saved transitions
-> RL trains cheaply without more LLM calls
```


# 2. Imports and Configuration

Do not hardcode API keys in notebooks. Use a `.env` file or environment variables.

This setup cell keeps LLM support optional. If the API key or OpenAI client package is missing, the notebook still runs the handcoded RL sections.

In [ ]:
# Optional install cell. Run only if these packages are missing.
# %pip install openai python-dotenv pandas matplotlib

In [ ]:
import os
import json
import random
import time
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed; plotting cells will be skipped.")

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False
    print("python-dotenv is not installed; reading environment variables directly.")

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

random.seed(42)
np.random.seed(42)

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH)

KIMI_API_KEY = os.getenv("KIMI_API_KEY")
KIMI_BASE_URL = os.getenv("KIMI_BASE_URL", "https://api.moonshot.ai/v1")
KIMI_MODEL = os.getenv("KIMI_MODEL", "moonshot-v1-8k")

LLM_ENABLED = bool(KIMI_API_KEY and OpenAI is not None)

if LLM_ENABLED:
    client = OpenAI(api_key=KIMI_API_KEY, base_url=KIMI_BASE_URL)
    print(f"LLM enabled with model: {KIMI_MODEL}")
else:
    client = None
    print("LLM disabled: set KIMI_API_KEY and install openai to enable LLM cells.")

# 3. State Space and Action Space

The state is deliberately small so the RL mechanics stay interpretable. This is a toy environment for learning RL system design, not a realistic model of dating.

In [ ]:
INTERESTS = ["low", "medium", "high"]
STAGES = ["opener", "chat", "date"]
TONES = ["cold", "neutral", "warm"]

ACTIONS = [
    "ask_question",
    "give_compliment",
    "share_story",
    "be_playful",
    "be_direct",
    "suggest_date",
    "slow_down",
    "end_chat",
]

ACTION_GUIDANCE = {
    "ask_question": "Ask an open-ended question to keep the conversation moving.",
    "give_compliment": "Give a specific, light compliment without overdoing it.",
    "share_story": "Share a short personal story that builds rapport.",
    "be_playful": "Use playful banter or gentle teasing.",
    "be_direct": "Use direct but warm escalation.",
    "suggest_date": "Suggest a simple, low-pressure plan to meet.",
    "slow_down": "Reduce pressure and rebuild comfort.",
    "end_chat": "End the conversation.",
}

N_STATES = len(INTERESTS) * len(STAGES) * len(TONES)
N_ACTIONS = len(ACTIONS)


def _label_to_id(value, labels, name):
    if isinstance(value, str):
        if value not in labels:
            raise ValueError(f"Invalid {name} label {value!r}. Expected one of {labels}.")
        return labels.index(value)
    value = int(value)
    if not 0 <= value < len(labels):
        raise ValueError(f"Invalid {name} index {value}. Expected 0 to {len(labels) - 1}.")
    return value


def state_to_id(interest, stage, tone):
    interest_id = _label_to_id(interest, INTERESTS, "interest")
    stage_id = _label_to_id(stage, STAGES, "stage")
    tone_id = _label_to_id(tone, TONES, "tone")
    return interest_id * len(STAGES) * len(TONES) + stage_id * len(TONES) + tone_id


def id_to_state(state_id):
    state_id = int(state_id)
    if not 0 <= state_id < N_STATES:
        raise ValueError(f"Invalid state_id {state_id}. Expected 0 to {N_STATES - 1}.")
    interest = state_id // (len(STAGES) * len(TONES))
    remainder = state_id % (len(STAGES) * len(TONES))
    stage = remainder // len(TONES)
    tone = remainder % len(TONES)
    return interest, stage, tone


def describe_state(state_id):
    interest, stage, tone = id_to_state(state_id)
    return f"interest={INTERESTS[interest]}, stage={STAGES[stage]}, tone={TONES[tone]}"


def action_to_id(action_name):
    if action_name not in ACTIONS:
        raise ValueError(f"Invalid action {action_name!r}. Expected one of {ACTIONS}.")
    return ACTIONS.index(action_name)


print(f"States: {N_STATES}")
print(f"Actions: {N_ACTIONS}")

# 4. Reward Function

The reward function is the learning signal. The Q-table learns to prefer actions that lead to higher expected future reward.

The reward is separate from the simulator: environments produce transitions, and this function scores the resulting state/outcome.

In [ ]:
def compute_reward(next_state_id, outcome=None):
    """Score a transition from its next state and terminal outcome."""
    if outcome == "date_success":
        return 30.0
    if outcome in {"date_fail", "date_failed", "ghosted", "ended", "ended_by_agent"}:
        return -5.0

    interest, stage, tone = id_to_state(next_state_id)

    interest_score = [-0.40, 0.20, 0.80][interest]
    stage_score = [0.00, 0.30, 0.80][stage]
    tone_score = [-0.30, 0.10, 0.40][tone]
    step_penalty = -0.10

    return interest_score + stage_score + tone_score + step_penalty

# 5. Policy Helpers

A policy chooses an action from a state. Random and rule-based policies are baselines; the learned policies come from Q-tables.

In [ ]:
def choose_action(Q, state_id, epsilon):
    if random.random() < epsilon:
        return random.randrange(N_ACTIONS)
    return int(np.argmax(Q[state_id]))


def greedy_action(Q, state_id):
    return int(np.argmax(Q[state_id]))


def random_policy(state_id):
    return random.randrange(N_ACTIONS)


def rule_based_policy(state_id):
    """Simple hand-written baseline policy."""
    interest, stage, tone = id_to_state(state_id)

    if interest == 2 and stage == 2 and tone == 2:
        return action_to_id("suggest_date")
    if tone == 0:
        return action_to_id("slow_down")
    if interest == 0:
        return action_to_id("ask_question")
    if stage == 0:
        return action_to_id("ask_question")
    if interest == 2 and tone == 2:
        return action_to_id("be_direct")
    if tone == 2:
        return action_to_id("be_playful")
    return action_to_id("share_story")

# 6. Environment 1: Fast Handcoded Simulator

This is the cheap baseline environment. It is not realistic. It exists so we can train thousands of episodes quickly and inspect the RL mechanics.

It returns exactly:

```python
next_state_id, reward, done, outcome
```

In [ ]:
def _clamp(value, low=0, high=2):
    return max(low, min(high, int(value)))


def simulator_step_hardcoded(state_id, action_id):
    """Fast toy simulator. No LLM calls happen here."""
    interest, stage, tone = id_to_state(state_id)
    action_name = ACTIONS[int(action_id)]
    done = False
    outcome = None

    if action_name == "ask_question":
        interest += np.random.choice([0, 1], p=[0.55, 0.45])
        tone += np.random.choice([0, 1], p=[0.65, 0.35])
        if interest >= 1 and tone >= 1:
            stage += np.random.choice([0, 1], p=[0.70, 0.30])

    elif action_name == "give_compliment":
        if tone == 0:
            interest -= 1
            tone -= 1
        else:
            interest += np.random.choice([0, 1], p=[0.45, 0.55])
            tone += np.random.choice([0, 1], p=[0.55, 0.45])

    elif action_name == "share_story":
        if interest >= 1:
            stage += np.random.choice([0, 1], p=[0.50, 0.50])
            tone += np.random.choice([0, 1], p=[0.60, 0.40])
        else:
            interest += np.random.choice([0, 1], p=[0.75, 0.25])

    elif action_name == "be_playful":
        if tone >= 1 and interest >= 1:
            interest += np.random.choice([0, 1], p=[0.55, 0.45])
            tone += 1
        else:
            interest -= 1
            tone -= 1

    elif action_name == "be_direct":
        if interest >= 2 and tone >= 1:
            stage += 1
            tone += np.random.choice([0, 1], p=[0.65, 0.35])
        elif interest >= 1:
            stage += np.random.choice([0, 1], p=[0.70, 0.30])
        else:
            interest -= 1
            tone -= 1

    elif action_name == "suggest_date":
        done = True
        if interest == 2 and stage == 2 and tone == 2:
            outcome = "date_success"
        elif interest == 2 and stage >= 1 and tone >= 1 and random.random() < 0.35:
            outcome = "date_success"
        else:
            outcome = "date_fail"

    elif action_name == "slow_down":
        tone += np.random.choice([0, 1], p=[0.55, 0.45])
        if interest == 0:
            interest += np.random.choice([0, 1], p=[0.70, 0.30])

    elif action_name == "end_chat":
        done = True
        outcome = "ended"

    interest = _clamp(interest)
    stage = _clamp(stage)
    tone = _clamp(tone)

    if not done and interest == 0 and tone == 0 and random.random() < 0.08:
        done = True
        outcome = "ghosted"

    next_state_id = state_to_id(interest, stage, tone)
    reward = compute_reward(next_state_id, outcome=outcome)
    return next_state_id, reward, done, outcome

# 7. Reset Function

Episodes start in plausible opening states. This is separate so experiments can swap in a different reset distribution later.

In [ ]:
def reset_episode():
    interest = np.random.choice([0, 1], p=[0.35, 0.65])
    stage = 0
    tone = np.random.choice([0, 1], p=[0.25, 0.75])
    return state_to_id(interest, stage, tone)

# 8. Generic Q-learning Episode Runner

This is where Q-learning actually happens when `learn=True`. There is only one Q-learning episode runner, and it accepts any environment function.

In [ ]:
def run_episode_q_learning(
    env_step,
    Q=None,
    policy_fn=None,
    epsilon=0.0,
    max_steps=25,
    reset_fn=reset_episode,
    learn=False,
    alpha=0.1,
    gamma=0.95,
    trace=False,
):
    state_id = reset_fn()
    total_reward = 0.0
    trace_rows = []
    outcome = "max_steps"

    for step in range(max_steps):
        if policy_fn is not None:
            action_id = int(policy_fn(state_id))
        elif Q is not None:
            action_id = choose_action(Q, state_id, epsilon)
        else:
            action_id = random_policy(state_id)

        next_state_id, reward, done, outcome = env_step(state_id, action_id)
        total_reward += reward

        if learn:
            # THIS IS WHERE THE RL AGENT TRAINS.
            # The Q-table is the trained model.
            best_next = np.max(Q[next_state_id])
            target = reward + (0.0 if done else gamma * best_next)
            Q[state_id, action_id] += alpha * (target - Q[state_id, action_id])

        if trace:
            trace_rows.append({
                "step": step,
                "state_id": state_id,
                "state": describe_state(state_id),
                "action_id": action_id,
                "action": ACTIONS[action_id],
                "next_state_id": next_state_id,
                "next_state": describe_state(next_state_id),
                "reward": reward,
                "done": done,
                "outcome": outcome,
            })

        state_id = next_state_id
        if done:
            break

    return total_reward, step + 1, outcome, trace_rows

# 9. Q-learning Training

`alpha` is the learning rate, `gamma` is the discount factor, and `epsilon` is the exploration rate. The Q-table is updated inside `run_episode_q_learning`.

In [ ]:
def train_q_learning(
    env_step,
    episodes=8000,
    reset_fn=reset_episode,
    alpha=0.1,
    gamma=0.95,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay_fraction=0.75,
):
    Q = np.zeros((N_STATES, N_ACTIONS))
    rewards = []
    outcomes = []

    for episode in range(episodes):
        frac = min(1.0, episode / max(1, int(episodes * epsilon_decay_fraction)))
        epsilon = epsilon_start + frac * (epsilon_end - epsilon_start)

        reward, steps, outcome, _ = run_episode_q_learning(
            env_step=env_step,
            Q=Q,
            epsilon=epsilon,
            reset_fn=reset_fn,
            learn=True,
            alpha=alpha,
            gamma=gamma,
        )

        rewards.append(reward)
        outcomes.append(outcome)

    return Q, rewards, outcomes


Q_hardcoded, rewards_hardcoded, outcomes_hardcoded = train_q_learning(
    env_step=simulator_step_hardcoded,
    episodes=8000,
)

print("Q-learning trained on handcoded simulator.")

# 10. SARSA Training

Q-learning is off-policy: it updates toward the greedy next action. SARSA is on-policy: it updates toward the action the policy actually takes next. Both are tabular RL methods.

In [ ]:
def run_episode_sarsa(
    env_step,
    Q,
    epsilon=0.0,
    max_steps=25,
    reset_fn=reset_episode,
    learn=False,
    alpha=0.1,
    gamma=0.95,
):
    state_id = reset_fn()
    action_id = choose_action(Q, state_id, epsilon)
    total_reward = 0.0
    outcome = "max_steps"

    for step in range(max_steps):
        next_state_id, reward, done, outcome = env_step(state_id, action_id)
        total_reward += reward

        if done:
            if learn:
                Q[state_id, action_id] += alpha * (reward - Q[state_id, action_id])
            break

        next_action_id = choose_action(Q, next_state_id, epsilon)

        if learn:
            # SARSA trains the same table shape, but with an on-policy TD target.
            target = reward + gamma * Q[next_state_id, next_action_id]
            Q[state_id, action_id] += alpha * (target - Q[state_id, action_id])

        state_id = next_state_id
        action_id = next_action_id

    return total_reward, step + 1, outcome


def train_sarsa(
    env_step,
    episodes=8000,
    reset_fn=reset_episode,
    alpha=0.1,
    gamma=0.95,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay_fraction=0.75,
):
    Q = np.zeros((N_STATES, N_ACTIONS))
    rewards = []
    outcomes = []

    for episode in range(episodes):
        frac = min(1.0, episode / max(1, int(episodes * epsilon_decay_fraction)))
        epsilon = epsilon_start + frac * (epsilon_end - epsilon_start)

        reward, steps, outcome = run_episode_sarsa(
            env_step=env_step,
            Q=Q,
            epsilon=epsilon,
            reset_fn=reset_fn,
            learn=True,
            alpha=alpha,
            gamma=gamma,
        )

        rewards.append(reward)
        outcomes.append(outcome)

    return Q, rewards, outcomes


Q_sarsa_hardcoded, rewards_sarsa_hardcoded, outcomes_sarsa_hardcoded = train_sarsa(
    env_step=simulator_step_hardcoded,
    episodes=8000,
)

print("SARSA trained on handcoded simulator.")

# 11. Evaluation Utilities

All rows in this first table are evaluated on the handcoded simulator. This makes the random, rule-based, Q-learning, and SARSA policies comparable.

In [ ]:
def make_greedy_q_policy(Q):
    return lambda state_id: greedy_action(Q, state_id)


def evaluate_policy(env_step, policy_fn, episodes=1000, reset_fn=reset_episode):
    rewards = []
    steps_list = []
    outcomes = []

    for _ in range(episodes):
        reward, steps, outcome, _ = run_episode_q_learning(
            env_step=env_step,
            policy_fn=policy_fn,
            reset_fn=reset_fn,
            learn=False,
        )
        rewards.append(reward)
        steps_list.append(steps)
        outcomes.append(outcome)

    outcome_counts = dict(Counter(outcomes))
    success_rate = outcome_counts.get("date_success", 0) / episodes
    return {
        "avg_reward": float(np.mean(rewards)),
        "success_rate": float(success_rate),
        "avg_steps": float(np.mean(steps_list)),
        "outcome_counts": outcome_counts,
    }


def evaluation_row(policy_name, training_env, evaluation_env, result):
    return {
        "policy_name": policy_name,
        "training_env": training_env,
        "evaluation_env": evaluation_env,
        "avg_reward": result["avg_reward"],
        "success_rate": result["success_rate"],
        "avg_steps": result["avg_steps"],
        "outcome_counts": result["outcome_counts"],
    }


baseline_rows = [
    evaluation_row("random_policy", None, "handcoded", evaluate_policy(simulator_step_hardcoded, random_policy)),
    evaluation_row("rule_based_policy", None, "handcoded", evaluate_policy(simulator_step_hardcoded, rule_based_policy)),
    evaluation_row("Q_hardcoded", "handcoded", "handcoded", evaluate_policy(simulator_step_hardcoded, make_greedy_q_policy(Q_hardcoded))),
    evaluation_row("Q_sarsa_hardcoded", "handcoded", "handcoded", evaluate_policy(simulator_step_hardcoded, make_greedy_q_policy(Q_sarsa_hardcoded))),
]

baseline_results = pd.DataFrame(baseline_rows)
baseline_results

# 12. Plot Learning Curves

This plot is optional. It smooths episode rewards so the broad learning trend is easier to see.

In [ ]:
def moving_average(values, window=100):
    values = np.asarray(values, dtype=float)
    if len(values) < window:
        return values
    weights = np.ones(window) / window
    return np.convolve(values, weights, mode="valid")


if plt is None:
    print("Skipping plot because matplotlib is not installed.")
else:
    plt.figure(figsize=(10, 4))
    plt.plot(moving_average(rewards_hardcoded), label="Q-learning")
    plt.plot(moving_average(rewards_sarsa_hardcoded), label="SARSA")
    plt.title("Handcoded Simulator Learning Curves")
    plt.xlabel("Episode")
    plt.ylabel("Moving average reward")
    plt.legend()
    plt.show()

# 13. LLM Classifier Interface

This is not training. This is the interface layer: it maps real text into the discrete RL state so an already-trained Q-table can recommend an abstract action.

Known labeling convention: without explicit opener context, a short reply like `"k"` should be treated as `low/chat/cold`, not `low/opener/cold`.

In [ ]:
INTEREST_MAP = {label: i for i, label in enumerate(INTERESTS)}
STAGE_MAP = {label: i for i, label in enumerate(STAGES)}
TONE_MAP = {label: i for i, label in enumerate(TONES)}

CLASSIFIER_SYSTEM_PROMPT = """You are a careful classifier for a toy dating-app RL state space.
Return ONLY valid JSON with exactly these fields:
- "interest": one of "low", "medium", "high"
- "stage": one of "opener", "chat", "date"
- "tone": one of "cold", "neutral", "warm"

Definitions:
- low interest: short, dry, no questions back, no curiosity
- medium interest: engaged but not effusive, some response effort
- high interest: enthusiastic, asks questions back, positive emotion
- opener stage: first 1-2 exchanges or clear first message
- chat stage: ongoing back-and-forth or a reply without clear opener context
- date stage: discussing meeting up or strong rapport/flirtation
- cold tone: terse, formal, low warmth
- neutral tone: cordial but not especially warm
- warm tone: friendly, playful, joking, genuinely enthusiastic

Important: if the message is just "k" and no context says it is the first message, classify it as low/chat/cold.
JSON only. No explanation."""


def require_llm():
    if not LLM_ENABLED:
        raise RuntimeError("LLM disabled. Set KIMI_API_KEY and install openai to enable LLM cells.")


def validate_state_labels(payload):
    expected = {"interest", "stage", "tone"}
    if set(payload) != expected:
        raise ValueError(f"Expected keys {expected}, got {payload}")
    if payload["interest"] not in INTEREST_MAP:
        raise ValueError(f"Invalid interest label: {payload}")
    if payload["stage"] not in STAGE_MAP:
        raise ValueError(f"Invalid stage label: {payload}")
    if payload["tone"] not in TONE_MAP:
        raise ValueError(f"Invalid tone label: {payload}")


def classify_message_with_llm(message, conversation_context=None):
    require_llm()
    if conversation_context:
        context_text = json.dumps(conversation_context, ensure_ascii=False, indent=2)
    else:
        context_text = "No prior context. Treat very short replies as chat unless clearly an opener."

    user_prompt = f"Conversation context:\n{context_text}\n\nMessage to classify:\n{message}"
    response = client.chat.completions.create(
        model=KIMI_MODEL,
        messages=[
            {"role": "system", "content": CLASSIFIER_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    payload = json.loads(response.choices[0].message.content)
    validate_state_labels(payload)
    return payload


def state_from_message(message, conversation_context=None):
    result = classify_message_with_llm(message, conversation_context=conversation_context)
    return state_to_id(result["interest"], result["stage"], result["tone"])


def recommend_action_from_message(message, Q=Q_hardcoded, conversation_context=None):
    state_id = state_from_message(message, conversation_context=conversation_context)
    action_id = greedy_action(Q, state_id)
    return {
        "message": message,
        "state_id": state_id,
        "state": describe_state(state_id),
        "action_id": action_id,
        "action": ACTIONS[action_id],
    }


RUN_LLM_CLASSIFIER_TESTS = False

if RUN_LLM_CLASSIFIER_TESTS and LLM_ENABLED:
    classifier_tests = [
        ("hey", {"interest": "low", "stage": "opener", "tone": "neutral"}),
        ("k", {"interest": "low", "stage": "chat", "tone": "cold"}),
        ("yeah I love hiking too! where do you usually go?", {"interest": "high", "stage": "chat", "tone": "warm"}),
    ]
    for message, expected in classifier_tests:
        got = classify_message_with_llm(message)
        print(message, "expected=", expected, "got=", got)
else:
    print("Skipping classifier API tests. Set RUN_LLM_CLASSIFIER_TESTS=True intentionally to run them.")

# 14. Environment 2: LLM World Model / Transition Generator

The LLM world model is **not** the trained model. It only generates simulated transitions that can later be cached.

Warning: this section can call the API. Do not run transition generation casually.

In [ ]:
VALID_OUTCOMES = {None, "date_success", "date_fail", "ghosted", "ended", "max_steps"}


def validate_llm_transition_payload(payload):
    required = {"response_text", "next_interest", "next_stage", "next_tone", "done", "outcome"}
    missing = required - set(payload)
    if missing:
        raise ValueError(f"Missing transition keys {missing}: {payload}")
    if payload["next_interest"] not in INTERESTS:
        raise ValueError(f"Invalid next_interest: {payload}")
    if payload["next_stage"] not in STAGES:
        raise ValueError(f"Invalid next_stage: {payload}")
    if payload["next_tone"] not in TONES:
        raise ValueError(f"Invalid next_tone: {payload}")
    if not isinstance(payload["done"], bool):
        raise ValueError(f"done must be boolean: {payload}")
    if payload["outcome"] not in VALID_OUTCOMES:
        raise ValueError(f"Invalid outcome: {payload}")


def generate_llm_transition(state_id, action_id, history=None, temperature=0.3, max_retries=1):
    """Generate one simulated transition with the LLM and return a validated record."""
    require_llm()

    state_id = int(state_id)
    action_id = int(action_id)
    interest_id, stage_id, tone_id = id_to_state(state_id)
    action_name = ACTIONS[action_id]

    if action_name == "end_chat":
        outcome = "ended"
        return {
            "state_id": state_id,
            "state": describe_state(state_id),
            "action_id": action_id,
            "action": action_name,
            "next_state_id": state_id,
            "next_state": describe_state(state_id),
            "reward": compute_reward(state_id, outcome=outcome),
            "done": True,
            "outcome": outcome,
            "response_text": "",
            "reason": "Agent ended the chat; no LLM call was needed.",
        }

    system_prompt = """You are simulating the OTHER person in a realistic dating-app conversation for a toy RL environment.
You are not trying to be overly positive. Preserve realistic uncertainty, skepticism, neutral responses, and occasional negative outcomes.
Premature date suggestions should often fail unless rapport is genuinely strong.
Return ONLY valid JSON. No markdown and no explanation outside JSON."""

    history_text = json.dumps(history or [], ensure_ascii=False, indent=2)
    user_prompt = f"""Current discrete state:
- interest: {INTERESTS[interest_id]}
- stage: {STAGES[stage_id]}
- tone: {TONES[tone_id]}

Agent abstract action:
- {action_name}: {ACTION_GUIDANCE[action_name]}

Conversation history:
{history_text}

Simulate the other person's next 1-2 sentence response and resulting next state.
Return JSON with exactly these fields:
{{
  "response_text": "short realistic reply",
  "next_interest": "low|medium|high",
  "next_stage": "opener|chat|date",
  "next_tone": "cold|neutral|warm",
  "done": false,
  "outcome": null,
  "reason": "brief reason"
}}

Allowed outcomes: null, "date_success", "date_fail", "ghosted", "ended", "max_steps".
Use null unless the conversation ends this turn."""

    last_error = None
    for attempt in range(max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=KIMI_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=temperature,
                response_format={"type": "json_object"},
            )
            payload = json.loads(response.choices[0].message.content)
            validate_llm_transition_payload(payload)
            break
        except Exception as exc:
            last_error = exc
            if attempt < max_retries:
                print(f"LLM transition invalid or failed on attempt {attempt + 1}; retrying...")
                time.sleep(1.0)
            else:
                raise RuntimeError(f"LLM transition failed after {max_retries + 1} attempts: {last_error}") from exc

    next_state_id = state_to_id(payload["next_interest"], payload["next_stage"], payload["next_tone"])
    outcome = payload["outcome"]
    reward = compute_reward(next_state_id, outcome=outcome)

    return {
        "state_id": state_id,
        "state": describe_state(state_id),
        "action_id": action_id,
        "action": action_name,
        "next_state_id": next_state_id,
        "next_state": describe_state(next_state_id),
        "reward": reward,
        "done": bool(payload["done"]),
        "outcome": outcome,
        "response_text": payload["response_text"],
        "reason": payload.get("reason", ""),
    }

# 15. Cached LLM Transition Dataset

This is the key workflow for avoiding wasted prompts.

Do **not** train by repeatedly calling the LLM inside the episode loop. Generate transitions once, save them to JSONL, then train from the cache.

Full cache size:

```text
27 states x 8 actions x 1 sample = 216 LLM calls
27 states x 8 actions x 3 samples = 648 LLM calls
```

Start small.

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

CACHE_PATH = DATA_DIR / "llm_transition_cache.jsonl"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"LLM transition cache: {CACHE_PATH}")


def save_jsonl(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def append_jsonl(record, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path):
    path = Path(path)

    if not path.exists():
        return []

    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    return records


def generate_transition_cache(
    samples_per_state_action=1,
    state_ids=None,
    action_ids=None,
    cache_path=CACHE_PATH,
    sleep_between_calls=1.0,
    resume=True,
    allow_full_generation=False,
):
    if state_ids is None:
        if not allow_full_generation:
            raise ValueError(
                "Refusing to generate transitions for all states by default. "
                "Pass explicit state_ids, or set allow_full_generation=True."
            )
        state_ids = list(range(N_STATES))

    if action_ids is None:
        if not allow_full_generation:
            raise ValueError(
                "Refusing to generate transitions for all actions by default. "
                "Pass explicit action_ids, or set allow_full_generation=True."
            )
        action_ids = list(range(N_ACTIONS))

    if not LLM_ENABLED:
        print("LLM disabled. Set KIMI_API_KEY to generate transitions.")
        return load_jsonl(cache_path)

    existing_records = load_jsonl(cache_path) if resume else []
    existing_keys = {r.get("cache_key") for r in existing_records}

    total_needed = len(state_ids) * len(action_ids) * samples_per_state_action
    made = 0
    print(f"Transition cache request: {total_needed} possible records.")

    for state_id in state_ids:
        for action_id in action_ids:
            for sample_idx in range(samples_per_state_action):
                cache_key = f"{state_id}:{action_id}:{sample_idx}"
                if cache_key in existing_keys:
                    continue

                record = generate_llm_transition(state_id, action_id)
                record["cache_key"] = cache_key
                record["sample_idx"] = sample_idx
                record["created_at"] = time.time()

                append_jsonl(record, cache_path)
                existing_records.append(record)
                existing_keys.add(cache_key)
                made += 1

                if made % 10 == 0:
                    print(f"Generated {made} new transitions...")

                time.sleep(sleep_between_calls)

    print(f"Done. Added {made} new transitions. Total records: {len(existing_records)}")
    return existing_records


## Safe Cache Demo Cell

This cell is intentionally opt-in. Set `RUN_LLM_CACHE_DEMO = True` only when you want to spend a couple of API calls.

Full cache generation now requires `allow_full_generation=True`, so a bare call to `generate_transition_cache()` will refuse to generate all state-action pairs by accident.


In [ ]:
RUN_LLM_CACHE_DEMO = False

if RUN_LLM_CACHE_DEMO:
    if not LLM_ENABLED:
        print("Skipping LLM cache demo because LLM is disabled.")
    else:
        demo_state = state_to_id("medium", "chat", "neutral")
        demo_actions = [
            action_to_id("ask_question"),
            action_to_id("be_playful"),
        ]

        demo_records = generate_transition_cache(
            samples_per_state_action=1,
            state_ids=[demo_state],
            action_ids=demo_actions,
            sleep_between_calls=1.0,
            resume=True,
        )
else:
    print("Skipping LLM cache demo. Set RUN_LLM_CACHE_DEMO=True intentionally to run it.")


In [ ]:
# INTENTIONAL FULL CACHE GENERATION — uncomment only if you understand the cost.
# This can make 27 * 8 * samples_per_state_action LLM calls.
#
# full_records = generate_transition_cache(
#     samples_per_state_action=1,
#     allow_full_generation=True,
#     sleep_between_calls=1.0,
#     resume=True,
# )


# 16. Cached LLM Environment

The cached environment samples from saved LLM-generated transitions. This allows thousands of RL training episodes with zero additional LLM calls.

If cache coverage is low, the cached environment is actually a hybrid: cached LLM transitions for covered state-action pairs, and handcoded simulator fallback for missing pairs.


In [ ]:
def cache_coverage_report(records):
    """
    Report how much of the state-action space is covered by cached LLM transitions.

    A full tabular environment has N_STATES * N_ACTIONS possible state-action pairs.
    If coverage is low, cached-LLM training will mostly use the handcoded fallback.
    """
    covered_pairs = {
        (int(r["state_id"]), int(r["action_id"]))
        for r in records
        if "state_id" in r and "action_id" in r
    }

    total_pairs = N_STATES * N_ACTIONS
    covered = len(covered_pairs)
    coverage = covered / total_pairs if total_pairs else 0.0

    print(f"Cached state-action pairs: {covered}/{total_pairs} ({coverage:.1%})")
    print(f"Total cached transition records: {len(records)}")

    if coverage == 0:
        print("No cached transitions found. Cached-LLM training will not run unless records are generated.")
    elif coverage < 0.10:
        print("Warning: cache coverage is very low. Training will mostly use the handcoded fallback.")
    elif coverage < 0.50:
        print("Warning: cache coverage is partial. Training will use a hybrid of cached LLM transitions and handcoded fallback.")
    else:
        print("Cache coverage is substantial. Cached-LLM training will rely more heavily on LLM-generated transitions.")

    return coverage


def build_transition_index(records):
    index = defaultdict(list)
    for record in records:
        key = (int(record["state_id"]), int(record["action_id"]))
        index[key].append(record)
    return index


def make_cached_llm_env_step(transition_index, fallback_env_step=simulator_step_hardcoded):
    def env_step(state_id, action_id):
        candidates = transition_index.get((int(state_id), int(action_id)), [])
        if candidates:
            record = random.choice(candidates)
            return (
                int(record["next_state_id"]),
                float(record["reward"]),
                bool(record["done"]),
                record.get("outcome"),
            )
        return fallback_env_step(state_id, action_id)

    return env_step


# 17. Train on Cached LLM Environment

This cell trains only if cached records exist. Training here does not call the LLM; it samples saved JSONL transitions.

`Q_cached_llm` is trained on the cached LLM environment. If cache coverage is incomplete, that environment falls back to the handcoded simulator for missing state-action pairs. Therefore, with a small cache, `Q_cached_llm` is better understood as a hybrid cached-LLM + handcoded-fallback policy.


In [ ]:
records = load_jsonl(CACHE_PATH)
coverage = cache_coverage_report(records)

Q_cached_llm = None
rewards_cached_llm = []
outcomes_cached_llm = []

if records:
    transition_index = build_transition_index(records)
    simulator_step_cached_llm = make_cached_llm_env_step(transition_index)

    Q_cached_llm, rewards_cached_llm, outcomes_cached_llm = train_q_learning(
        env_step=simulator_step_cached_llm,
        episodes=8000,
    )
    print("Q_cached_llm trained from cached transitions plus handcoded fallback for missing pairs.")
else:
    simulator_step_cached_llm = None
    print("No cached LLM transitions found. Generate cache first or use Q_hardcoded.")


# 18. Compare Handcoded vs Cached-LLM Policies

Policies should be compared on the same evaluation environment. Results across different environments are not directly comparable.

Cached-LLM rows are included only when cached records exist and `Q_cached_llm` has been trained. If `Q_cached_llm` is `None`, cached-policy evaluation is skipped so the notebook still runs top-to-bottom with no cache file.


In [ ]:
comparison_rows = []

comparison_specs = [
    ("random_policy", None, "handcoded", random_policy, simulator_step_hardcoded),
    ("rule_based_policy", None, "handcoded", rule_based_policy, simulator_step_hardcoded),
    ("Q_hardcoded", "handcoded", "handcoded", make_greedy_q_policy(Q_hardcoded), simulator_step_hardcoded),
    ("Q_sarsa_hardcoded", "handcoded", "handcoded", make_greedy_q_policy(Q_sarsa_hardcoded), simulator_step_hardcoded),
]

if Q_cached_llm is not None:
    comparison_specs.extend([
        ("Q_hardcoded", "handcoded", "cached_llm", make_greedy_q_policy(Q_hardcoded), simulator_step_cached_llm),
        ("Q_cached_llm", "cached_llm", "cached_llm", make_greedy_q_policy(Q_cached_llm), simulator_step_cached_llm),
        ("Q_cached_llm", "cached_llm", "handcoded", make_greedy_q_policy(Q_cached_llm), simulator_step_hardcoded),
    ])
else:
    print("Skipping cached-LLM policy evaluation because Q_cached_llm was not trained.")

for policy_name, training_env, evaluation_env, policy_fn, env_step in comparison_specs:
    result = evaluate_policy(env_step, policy_fn)
    comparison_rows.append(evaluation_row(policy_name, training_env, evaluation_env, result))

comparison_results = pd.DataFrame(comparison_rows)
comparison_results


# 19. Optional Appendix: Live LLM Training

This is slow and expensive because each RL step can make an API call. Use only for debugging. The preferred workflow is cached transitions.

The default example is 2 episodes and is opt-in.

In [ ]:
def make_live_llm_env_step(history_by_episode=None):
    """Tiny debugging environment that calls the LLM every step. Prefer cached transitions."""
    history = []

    def env_step(state_id, action_id):
        record = generate_llm_transition(state_id, action_id, history=history)
        history.append({
            "agent_action": ACTIONS[action_id],
            "other_response": record.get("response_text", ""),
        })
        return (
            record["next_state_id"],
            record["reward"],
            record["done"],
            record["outcome"],
        )

    return env_step


def train_q_learning_live_llm(episodes=2, **kwargs):
    if not LLM_ENABLED:
        print("LLM disabled. Live LLM training skipped.")
        return None, [], []
    print("Warning: live LLM training spends API calls inside the episode loop.")
    return train_q_learning(env_step=make_live_llm_env_step(), episodes=episodes, **kwargs)


RUN_LIVE_LLM_DEBUG = False

if RUN_LIVE_LLM_DEBUG:
    Q_live_llm_debug, rewards_live_llm_debug, outcomes_live_llm_debug = train_q_learning_live_llm(episodes=2)
else:
    print("Skipping live LLM training. Set RUN_LIVE_LLM_DEBUG=True intentionally to run 2 episodes.")

# 20. Final Summary

Current correct workflow:

1. Run the handcoded baseline training. This trains `Q_hardcoded` and `Q_sarsa_hardcoded` cheaply.

2. Keep all LLM flags off by default.

```python
RUN_LLM_CLASSIFIER_TESTS = False
RUN_LLM_CACHE_DEMO = False
RUN_LIVE_LLM_DEBUG = False
```

3. To use the LLM world model, generate cached transitions intentionally. Start with a tiny subset.

4. Inspect cache coverage. Low coverage means cached-LLM training mostly uses handcoded fallback.

5. Train `Q_cached_llm` only after cached records exist. This does not call the LLM again.

6. Use the LLM classifier only for real-message demo/interface. It does not train the RL agent.

7. Avoid live LLM training. It is slow and expensive. The preferred workflow is: LLM generates transitions once -> save JSONL cache -> train RL from cache.

What is trained:

- `Q_hardcoded`
- `Q_sarsa_hardcoded`
- optionally `Q_cached_llm`

Where LLM calls happen:

- `classify_message_with_llm`
- `generate_llm_transition`
- `generate_transition_cache`
- optional live LLM appendix

The LLM is not being trained. The Q-table is being trained. This notebook is for learning tabular RL system design with a simplified toy conversation environment.
